In [1]:
import pandas as pd
import checkdmarc
import time
from tqdm import tqdm
import random
from collections import Counter
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
CHECKPOINT_FILE = "dmarc_checkpoint.csv"
FINAL_FILE = "dmarc_final.csv"


In [3]:
def check_dmarc(domain):
    returned_results = {
        "isPresent": False,
        "policy": None,
        "sp": None,
        "pct": None,
        "rua": None,
        "ruf": None,
        "adkim": None,
        "aspf": None,
        "error": None,
        "valid": False,
    }
    results = checkdmarc.check_dmarc(domain)
    if ("record" not in results.keys()):
        returned_results["isPresent"] = False
        return returned_results
    if (results["record"] == None):
        returned_results["isPresent"] =  False
    else:
        returned_results["isPresent"] = True

    returned_results["error"] = results.get("error", "")
    returned_results["valid"] = results.get("valid")

    tags = results.get("tags", {})
    
    returned_results["policy"] = tags.get("p", {}).get("value")
    returned_results["sp"]  = tags.get("sp", {}).get("value")
    returned_results["adkim"] = tags.get("adkim", {}).get("value")
    returned_results["aspf"] = tags.get("aspf", {}).get("value")
    returned_results["pct"] = tags.get("pct", {}).get("value")
    rua_address = tags.get("rua", {}).get("value")
    
    if isinstance(rua_address, list) and len(rua_address) > 0:
        returned_results["rua"] = rua_address
    else:
        returned_results["rua"] = None
    

    ruf_address = tags.get("ruf", {}).get("value")
    
    if isinstance(ruf_address, list) and len(ruf_address) > 0:
        returned_results["ruf"] = ruf_address
    else:
        returned_results["ruf"] = None
    
    return returned_results

In [4]:
def process_data():
    df = pd.read_csv("tranco_VQPQN.csv")
    df.columns = ["index", "dns"]
    return df

In [5]:
error_counter = Counter()


In [6]:
def safe_check(dns):
    try:
        result = check_dmarc(dns)
        return dns, result, None
    except Exception as e:
        print(f"error occurred for {dns} - {e}")
        error_type = type(e).__name__
        error_counter[error_type] += 1
        return dns, None, error_type

In [7]:

def save_checkpoint(full_results):
    df = pd.DataFrame.from_dict(full_results, orient="index")
    df.to_csv(CHECKPOINT_FILE, index=True)
    print(f"Checkpoint saved ({len(df)} domains processed)")


In [ ]:

def main():
    df = process_data()
    file_exists = os.path.exists(CHECKPOINT_FILE)
    if file_exists:
        print("Loading checkpoint...")
        existing_df = pd.read_csv(CHECKPOINT_FILE, usecols=[0])
        processed = set(existing_df.iloc[:, 0])
    else:
        full_results = {}
        processed = set()

    error_counter = Counter()

    domains_to_process = [
        dns for dns in df["dns"] if dns not in processed
    ]
    
    full_results = {}
    error_counter = Counter()

    with ThreadPoolExecutor(max_workers=20) as executor:
        results = executor.map(safe_check, domains_to_process)

        for i, (dns, result, error) in enumerate(
            tqdm(results, total=len(domains_to_process)), start=1
        ):
            full_results[dns] = result

            if error:
                error_counter[error] += 1
                print("errors encountered")

            row_df = pd.DataFrame([result], index=[dns])
            
            row_df.to_csv(
                CHECKPOINT_FILE,
                mode="a",
                header=not file_exists,
                index=True,
            )

            file_exists = True

    if error_counter:
        print("Errors encountered:")
        print(error_counter)

    full_results_df = pd.DataFrame.from_dict(full_results, orient="index")
    full_results_df.to_csv(FINAL_FILE, index=True)

In [9]:
main()

100%|██████████| 999999/999999 [8:54:47<00:00, 31.16it/s]    
